# 🏦 Aurix Fintech — Data Analytics & Decision Intelligence System
**Analyst:** Mubinabegim  
**Platform:** Aurix — Digital Gold Trading (Buy/Sell/Save)  
**Goal:** End-to-end data analysis with business insights and ML layer

## Part 1: Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('✅ Libraries loaded successfully')

In [ ]:
# ── Simulate realistic Aurix transaction dataset ──
np.random.seed(42)
random.seed(42)

n_records = 180
n_users = 30

# User IDs — some users are more active (power users)
user_weights = np.random.dirichlet(np.ones(n_users) * 0.5)
user_ids = [f'USR_{str(i).zfill(3)}' for i in range(1, n_users + 1)]

# Generate timestamps over last 90 days
start_date = datetime(2025, 1, 1)
timestamps = [start_date + timedelta(
    days=random.randint(0, 89),
    hours=random.randint(0, 23),
    minutes=random.randint(0, 59)
) for _ in range(n_records)]

# Transaction types — buy-heavy (70/30 split, realistic for savings platform)
tx_types = np.random.choice(['buy', 'sell'], size=n_records, p=[0.68, 0.32])

# Gold amounts in grams (log-normal: most small, few large)
gold_amounts = np.round(np.random.lognormal(mean=1.2, sigma=0.8, size=n_records), 4)
gold_amounts = np.clip(gold_amounts, 0.1, 50.0)

# Gold price in EUR (realistic: ~60–65 EUR/gram in 2025, with daily variation)
base_price = 62.0
price_per_gram = base_price + np.random.normal(0, 1.5, n_records)
price_total = np.round(gold_amounts * price_per_gram, 2)

# Build DataFrame
df_raw = pd.DataFrame({
    'user_id': np.random.choice(user_ids, size=n_records, p=user_weights),
    'transaction_type': tx_types,
    'gold_amount_g': gold_amounts,
    'price_eur': price_total,
    'timestamp': timestamps
})

# ── Inject realistic data quality issues ──
# 5 duplicate rows
df_raw = pd.concat([df_raw, df_raw.sample(5)], ignore_index=True)
# 4 missing values
df_raw.loc[df_raw.sample(2).index, 'gold_amount_g'] = np.nan
df_raw.loc[df_raw.sample(2).index, 'price_eur'] = np.nan
# 2 negative/zero amounts (invalid)
df_raw.loc[df_raw.sample(2).index, 'gold_amount_g'] = -0.5

print(f'Raw dataset shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# ── Data Cleaning ──
print('=== Data Quality Report (Before Cleaning) ===')
print(f'Total records:      {len(df_raw)}')
print(f'Duplicate rows:     {df_raw.duplicated().sum()}')
print(f'Missing values:\n{df_raw.isnull().sum()}')
print(f'Negative amounts:   {(df_raw["gold_amount_g"] < 0).sum()}')

df = df_raw.copy()

# 1. Remove duplicates
df = df.drop_duplicates()

# 2. Drop rows with missing critical fields
df = df.dropna(subset=['gold_amount_g', 'price_eur'])

# 3. Remove invalid amounts
df = df[df['gold_amount_g'] > 0]

# 4. Sort by timestamp
df = df.sort_values('timestamp').reset_index(drop=True)

# 5. Add derived columns
df['date'] = df['timestamp'].dt.date
df['week'] = df['timestamp'].dt.to_period('W')
df['month'] = df['timestamp'].dt.to_period('M')
df['hour'] = df['timestamp'].dt.hour
df['price_per_gram'] = (df['price_eur'] / df['gold_amount_g']).round(2)

print(f'\n=== After Cleaning ===')
print(f'Clean records: {len(df)}')
print(f'Date range: {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
df.dtypes

## Part 2: Exploratory Data Analysis

In [ ]:
print('=== Core Statistics ===')
print(f"\nTotal buy volume:  {df[df.transaction_type=='buy']['gold_amount_g'].sum():.2f} g")
print(f"Total sell volume: {df[df.transaction_type=='sell']['gold_amount_g'].sum():.2f} g")
print(f"Total buy value:   €{df[df.transaction_type=='buy']['price_eur'].sum():,.2f}")
print(f"Total sell value:  €{df[df.transaction_type=='sell']['price_eur'].sum():,.2f}")
print(f"\nAvg transaction size: {df['gold_amount_g'].mean():.3f} g")
print(f"Median transaction:   {df['gold_amount_g'].median():.3f} g")
print(f"\nMost active users (top 5):")
print(df['user_id'].value_counts().head())

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# VISUALIZATION 1 — Buy vs Sell Overview
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Aurix — Transaction Overview', fontsize=16, fontweight='bold', y=1.02)

colors = {'buy': '#2ecc71', 'sell': '#e74c3c'}

# 1a. Transaction count
tx_counts = df['transaction_type'].value_counts()
axes[0].pie(tx_counts.values, labels=tx_counts.index.str.upper(),
            colors=[colors[x] for x in tx_counts.index],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Transaction Count Split', fontweight='bold')

# 1b. Volume (grams)
vol = df.groupby('transaction_type')['gold_amount_g'].sum()
bars = axes[1].bar(vol.index.str.upper(), vol.values,
                   color=[colors[x] for x in vol.index], width=0.5, edgecolor='white')
axes[1].set_title('Total Gold Volume (grams)', fontweight='bold')
axes[1].set_ylabel('Grams')
for bar, val in zip(bars, vol.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}g', ha='center', fontweight='bold')

# 1c. Value (EUR)
val_eur = df.groupby('transaction_type')['price_eur'].sum()
bars2 = axes[2].bar(val_eur.index.str.upper(), val_eur.values,
                    color=[colors[x] for x in val_eur.index], width=0.5, edgecolor='white')
axes[2].set_title('Total Value (EUR)', fontweight='bold')
axes[2].set_ylabel('EUR')
for bar, val in zip(bars2, val_eur.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'€{val:,.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('viz1_buy_vs_sell.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualization 1 saved')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# VISUALIZATION 2 — Trends Over Time
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Aurix — Transaction Trends Over Time', fontsize=16, fontweight='bold')

# Weekly volume by type
weekly = df.groupby(['week', 'transaction_type'])['gold_amount_g'].sum().unstack(fill_value=0)
weekly.index = weekly.index.astype(str)
weekly.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'], width=0.7, edgecolor='white')
axes[0].set_title('Weekly Gold Volume by Transaction Type', fontweight='bold')
axes[0].set_ylabel('Gold (grams)')
axes[0].set_xlabel('Week')
axes[0].legend(title='Type', labels=['BUY', 'SELL'])
axes[0].tick_params(axis='x', rotation=45)

# Daily transaction count (rolling 7-day)
daily_count = df.groupby('date').size().reset_index(name='count')
daily_count['date'] = pd.to_datetime(daily_count['date'])
daily_count = daily_count.set_index('date').resample('D').sum().fillna(0)
daily_count['rolling_7'] = daily_count['count'].rolling(7).mean()

axes[1].bar(daily_count.index, daily_count['count'], alpha=0.4, color='#3498db', label='Daily count')
axes[1].plot(daily_count.index, daily_count['rolling_7'], color='#2c3e50', lw=2.5, label='7-day rolling avg')
axes[1].set_title('Daily Transaction Count + 7-Day Rolling Average', fontweight='bold')
axes[1].set_ylabel('Number of Transactions')
axes[1].legend()

plt.tight_layout()
plt.savefig('viz2_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualization 2 saved')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# VISUALIZATION 3 — User Behavior
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Aurix — User Behavior Analysis', fontsize=16, fontweight='bold')

# Top 10 most active users
top_users = df['user_id'].value_counts().head(10)
sns.barplot(x=top_users.values, y=top_users.index, ax=axes[0],
            palette='Blues_r', orient='h')
axes[0].set_title('Top 10 Most Active Users', fontweight='bold')
axes[0].set_xlabel('Number of Transactions')

# Transaction size distribution
sns.histplot(data=df, x='gold_amount_g', hue='transaction_type',
             palette=colors, bins=25, ax=axes[1], alpha=0.7, edgecolor='white')
axes[1].set_title('Transaction Size Distribution', fontweight='bold')
axes[1].set_xlabel('Gold Amount (grams)')
axes[1].axvline(df['gold_amount_g'].median(), color='black', ls='--', lw=1.5, label=f'Median: {df["gold_amount_g"].median():.2f}g')
axes[1].legend()

# Hourly activity heatmap
hourly = df.groupby(['hour', 'transaction_type']).size().unstack(fill_value=0)
sns.heatmap(hourly.T, ax=axes[2], cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Transactions'})
axes[2].set_title('Hourly Activity Heatmap', fontweight='bold')
axes[2].set_xlabel('Hour of Day')

plt.tight_layout()
plt.savefig('viz3_user_behavior.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualization 3 saved')

## Part 3: Business Insights

In [ ]:
print('=' * 60)
print('AURIX — KEY BUSINESS INSIGHTS')
print('=' * 60)

buy_vol = df[df.transaction_type=='buy']['gold_amount_g'].sum()
sell_vol = df[df.transaction_type=='sell']['gold_amount_g'].sum()
net_accumulation = buy_vol - sell_vol

top_10_pct = df['user_id'].value_counts().head(3).sum() / len(df) * 100

buy_avg = df[df.transaction_type=='buy']['gold_amount_g'].mean()
sell_avg = df[df.transaction_type=='sell']['gold_amount_g'].mean()

peak_hour = df['hour'].value_counts().idxmax()

# Detect potential unusual activity (>3 std deviations)
mean_amt = df['gold_amount_g'].mean()
std_amt = df['gold_amount_g'].std()
unusual = df[df['gold_amount_g'] > mean_amt + 3 * std_amt]

print(f"""
📌 INSIGHT 1 — Strong Buy-Side Dominance
   Users buy significantly more gold than they sell.
   Net gold accumulation: +{net_accumulation:.2f} grams
   This indicates Aurix users treat digital gold as a savings vehicle
   (buy & hold), not a trading instrument. This is healthy for platform
   liquidity and aligns with Aurix's savings-first mission.

📌 INSIGHT 2 — Power User Concentration (Pareto Effect)
   Top 3 users account for {top_10_pct:.1f}% of all transactions.
   High-value user retention programs (VIP tiers, loyalty rewards)
   could significantly impact platform revenue.

📌 INSIGHT 3 — Sell Transactions Are Larger on Average
   Avg BUY size:  {buy_avg:.3f}g | Avg SELL size: {sell_avg:.3f}g
   Users buy small amounts frequently (micro-savings behavior)
   but sell in larger chunks (liquidation events). This suggests
   users accumulate patiently then exit strategically.

📌 INSIGHT 4 — Peak Activity at Hour {peak_hour}:00
   Most transactions occur at {peak_hour}:00. Marketing push notifications
   and price alerts sent before this peak window could increase
   conversion rates.

📌 INSIGHT 5 — {len(unusual)} Potentially Unusual Transactions Detected
   {len(unusual)} transactions exceed mean + 3σ threshold (>{mean_amt + 3*std_amt:.2f}g).
   These should be reviewed for compliance/AML purposes.
   Flagging large outlier transactions protects both users and platform.
""")

## Part 4: Intelligence Layer — User Classification (Option B)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# ── Build user-level feature matrix ──
user_features = df.groupby('user_id').agg(
    total_transactions=('transaction_type', 'count'),
    total_gold_bought=('gold_amount_g', lambda x: x[df.loc[x.index, 'transaction_type'] == 'buy'].sum()),
    total_gold_sold=('gold_amount_g', lambda x: x[df.loc[x.index, 'transaction_type'] == 'sell'].sum()),
    avg_transaction_size=('gold_amount_g', 'mean'),
    total_eur_value=('price_eur', 'sum'),
    unique_days=('date', 'nunique'),
).reset_index()

user_features['buy_ratio'] = user_features['total_gold_bought'] / (
    user_features['total_gold_bought'] + user_features['total_gold_sold'] + 1e-9)

# Label: ACTIVE = top 40% by transaction count
threshold = user_features['total_transactions'].quantile(0.6)
user_features['is_active'] = (user_features['total_transactions'] >= threshold).astype(int)

print(f'User feature matrix: {user_features.shape}')
print(f'Active users: {user_features["is_active"].sum()} / {len(user_features)}')
user_features.head()

In [ ]:
# ── Logistic Regression Classifier ──
feature_cols = ['total_transactions', 'avg_transaction_size', 'total_eur_value',
                'unique_days', 'buy_ratio']

X = user_features[feature_cols]
y = user_features['is_active']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(random_state=42)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)

print('=== Logistic Regression — Active User Classification ===')
print(classification_report(y_test, y_pred, target_names=['Inactive', 'Active']))

# Feature importance
coef_df = pd.DataFrame({'feature': feature_cols, 'coefficient': model.coef_[0]})
coef_df = coef_df.sort_values('coefficient', ascending=False)
print('\nFeature Importance (Coefficients):')
print(coef_df.to_string(index=False))

In [ ]:
# ── Visualize Classification Results ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('User Classification — Active vs Inactive', fontsize=14, fontweight='bold')

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Inactive', 'Active'],
            yticklabels=['Inactive', 'Active'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Feature coefficients
colors_coef = ['#2ecc71' if c > 0 else '#e74c3c' for c in coef_df['coefficient']]
axes[1].barh(coef_df['feature'], coef_df['coefficient'], color=colors_coef, edgecolor='white')
axes[1].axvline(0, color='black', lw=1)
axes[1].set_title('Feature Importance (Logistic Regression Coefficients)')
axes[1].set_xlabel('Coefficient Value')

plt.tight_layout()
plt.savefig('viz4_classification.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualization 4 saved')

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
APPROACH & LIMITATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Approach:
  • Built user-level feature aggregations from transaction data
  • Classified users as Active/Inactive using Logistic Regression
  • Features: transaction frequency, avg size, EUR value, unique active days, buy ratio

Limitations:
  • Small dataset (30 users) — real performance needs 1000+ users
  • Simulated data may not reflect true behavioral patterns
  • 'Active' label is rule-based, not from real business definition
  • No temporal features (recency, churn risk) included

Improvements:
  • Add RFM (Recency, Frequency, Monetary) segmentation
  • Use XGBoost for better performance on imbalanced data
  • Incorporate user profile data (age, region, account age)
""")

## Summary

| Component | Status |
|---|---|
| Data simulation & cleaning | ✅ |
| EDA + 4 visualizations | ✅ |
| Business insights (5) | ✅ |
| ML: Active user classifier | ✅ |
| Approach & limitations | ✅ |